# Data Mining_2_Module_3
## 01 — Time-Series Data Understanding & Preparation

This notebook reconstructs the **time-series preprocessing pipeline**.

### Final output
- unscaled cleaned time series → used by motifs/discords and classification;
- globally standardized copy → used only for unsupervised clustering.

In [1]:
from pathlib import Path
import warnings, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2"),
    Path("/Users/sarabiiavasco/Codice/DATA MINING/Progetto DM2"),
    Path("/Users/sarabiiavasco/Codice/DATA MINING/progetto dm2"),
]

def is_project_dir(path):
    return path.exists() and (
        (path / "data").exists()
        or (path / "artifacts").exists()
        or any(path.glob("CMI_timeseries_dataset*.pkl"))
    )

PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if is_project_dir(p)), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Project folder not found. Update PROJECT_CANDIDATES with your local project path."
    )

TS_PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "timeseries"
MODULE3_DIR = PROJECT_DIR / "artifacts" / "module3"

TS_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODULE3_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Processed time-series directory:", TS_PROCESSED_DIR)
print("Module 3 artifacts:", MODULE3_DIR)

Project directory: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2
Processed time-series directory: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2/data/processed/timeseries
Module 3 artifacts: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2/artifacts/module3


## 1. Load the raw time-series dataset

In [3]:
RAW_CANDIDATES = [
    PROJECT_DIR / "data" / "raw" / "CMI_timeseries_dataset.pkl.gz",
    PROJECT_DIR / "data" / "CMI_timeseries_dataset.pkl.gz",
    PROJECT_DIR / "CMI_timeseries_dataset.pkl.gz",
]

RAW_PATH = next((p for p in RAW_CANDIDATES if p.exists()), None)

if RAW_PATH is None:
    matches = [
        p for p in PROJECT_DIR.rglob("CMI_timeseries_dataset.pkl")
        if "processed" not in p.parts and "artifacts" not in p.parts
    ]
    if len(matches) == 1:
        RAW_PATH = matches[0]

if RAW_PATH is None:
    raise FileNotFoundError(
        "Raw time-series file not found. Put CMI_timeseries_dataset.pkl in data/raw/."
    )

df = pd.read_pickle(RAW_PATH)

print("Loaded:", RAW_PATH)
print("Object type:", type(df))
print("Number of time series:", len(df))
print("First shape:", df[0].shape)
display(df[0].head())

EXPECTED_COLUMNS = [
    "X", "Y", "Z", "enmo", "anglez", "non-wear_flag", "light",
    "battery_voltage", "weekday", "quarter", "relative_date_PCIAT",
    "id", "sii_binary"
]
missing = [c for c in EXPECTED_COLUMNS if c not in df[0].columns]
if missing:
    raise KeyError(f"Missing expected columns: {missing}")

Loaded: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2/CMI_timeseries_dataset.pkl.gz
Object type: <class 'list'>
Number of time series: 4437
First shape: (200, 13)


,X,Y,Z,enmo,anglez,non-wear_flag,light,battery_voltage,weekday,quarter,relative_date_PCIAT,id,sii_binary
0,-0.696639,-0.058477,-0.086717,0.051424,-6.090168,0.0,21.794674,4023.000000,4.0,4.0,253.0,2265,0
1,-0.479150,0.218840,-0.289497,0.062909,-19.893288,0.0,20.835835,4023.000000,4.0,4.0,253.0,2265,0
2,-0.440287,-0.171177,-0.326227,0.121546,-23.043303,0.0,6.294650,4023.000000,4.0,4.0,253.0,2265,0
3,-0.559035,-0.293204,-0.145227,0.084953,-9.694811,0.0,0.462963,4022.616699,4.0,4.0,253.0,2265,0
4,-0.531364,-0.210126,-0.086370,0.074157,-6.412807,0.0,4.058062,4020.583252,4.0,4.0,253.0,2265,0


## 2. Structure, missing values and participant/target consistency

In [4]:
lengths = pd.Series([len(ts) for ts in df], name="length")
all_data = pd.concat(df, ignore_index=True)

ts_level = pd.DataFrame({
    "id": [ts["id"].iloc[0] for ts in df],
    "sii_binary": [int(ts["sii_binary"].iloc[0]) for ts in df],
    "length": lengths,
})

user_level = (
    ts_level.groupby("id")
    .agg(n_time_series=("length","size"), target=("sii_binary","first"))
)

target_consistency = ts_level.groupby("id")["sii_binary"].nunique()

print("Time series:", len(df))
print("Rows:", len(all_data))
print("Columns:", len(all_data.columns))
print("Unique participants:", ts_level["id"].nunique())
print("Lengths:", sorted(lengths.unique().tolist()))
print("Total missing cells:", int(all_data.isna().sum().sum()))
print("Participants with inconsistent target:", int((target_consistency > 1).sum()))
print("\nTime series per participant:")
display(user_level["n_time_series"].describe().to_frame())
print("\nTime-series target distribution:")
display(ts_level["sii_binary"].value_counts(normalize=True).sort_index().rename("proportion").to_frame())

Time series: 4437
Rows: 887400
Columns: 13
Unique participants: 494
Lengths: [200]
Total missing cells: 0
Participants with inconsistent target: 0

Time series per participant:


,n_time_series
count,494.000000
mean,8.981781
std,10.485461
min,1.000000
25%,1.000000
50%,2.000000
75%,17.750000
max,36.000000



Time-series target distribution:


,proportion
sii_binary,
0,0.666441
1,0.333559


## 3. Context columns

`weekday`, `quarter` and `relative_date_PCIAT` are constant inside each 200-point series in the supplied dataset. They are treated as contextual metadata rather than dynamic signals.

One observation represents a **30-minute average**; this temporal resolution is used later to interpret motif windows.

In [5]:
context_cols = ["weekday", "quarter", "relative_date_PCIAT"]
context_summary = []

for col in context_cols:
    n_unique_inside = pd.Series([ts[col].nunique(dropna=True) for ts in df])
    context_summary.append({
        "column": col,
        "series_with_1_unique_value": int((n_unique_inside == 1).sum()),
        "series_with_more_than_1": int((n_unique_inside > 1).sum()),
    })

display(pd.DataFrame(context_summary))

,column,series_with_1_unique_value,series_with_more_than_1
0,weekday,4437,0
1,quarter,4437,0
2,relative_date_PCIAT,4437,0


## 4. Non-wear flag: diagnose intermediate values and binarize

In [6]:
intermediate_count = sum(
    ((ts["non-wear_flag"] > 0) & (ts["non-wear_flag"] < 1)).sum()
    for ts in df
)
total_rows = sum(len(ts) for ts in df)

print("Intermediate non-wear rows:", intermediate_count)
print("Percentage:", 100 * intermediate_count / total_rows)

# FINAL TRANSFORMATION:
# every intermediate non-zero value is treated as non-wear.
for ts in df:
    mask = (ts["non-wear_flag"] > 0) & (ts["non-wear_flag"] < 1)
    ts.loc[mask, "non-wear_flag"] = 1

print("\nValues after binarization:")
display(
    pd.concat([ts["non-wear_flag"] for ts in df], ignore_index=True)
    .value_counts()
    .sort_index()
    .to_frame("count")
)

Intermediate non-wear rows: 2116
Percentage: 0.23844940274960558

Values after binarization:


,count
non-wear_flag,
0.0,862305
1.0,25095


## 5. Z vs anglez redundancy

We found very strong redundancy between `Z` and `anglez`.  
`anglez` is therefore not retained among the final dynamic signals.

In [7]:
global_corr = all_data[["Z", "anglez"]].corr().iloc[0, 1]

within_corrs = []
for ts in df:
    c = np.corrcoef(ts["Z"].to_numpy(), ts["anglez"].to_numpy())[0, 1]
    if np.isfinite(c):
        within_corrs.append(c)

print("Global Pearson correlation Z-anglez:", global_corr)
print("Mean within-series correlation:", np.mean(within_corrs))
print("Median within-series correlation:", np.median(within_corrs))

Global Pearson correlation Z-anglez: 0.9931183028963252
Mean within-series correlation: 0.9884523073214381
Median within-series correlation: 0.9952280532434393


## 6. Outlier diagnostics

IQR, Hampel and Grubbs-style diagnostics were explored because sensor signals may contain abrupt changes.  
We **did not automatically remove all flagged points**, because many abrupt values are plausible activity changes.

Only the seven isolated extreme cells identified by the final decision rule are changed below.

In [8]:
def iqr_bounds(values, multiplier=1.5):
    values = np.asarray(values, dtype=float)
    q1, q3 = np.nanpercentile(values, [25, 75])
    iqr = q3 - q1
    return q1 - multiplier * iqr, q3 + multiplier * iqr

diagnostic_rows = []
for col in ["X", "Y", "Z", "enmo"]:
    values = np.concatenate([ts[col].to_numpy(dtype=float) for ts in df])
    lo15, hi15 = iqr_bounds(values, 1.5)
    lo3, hi3 = iqr_bounds(values, 3.0)
    diagnostic_rows.append({
        "signal": col,
        "min": np.nanmin(values),
        "max": np.nanmax(values),
        "IQR_1.5_flagged": int(((values < lo15) | (values > hi15)).sum()),
        "IQR_3_flagged": int(((values < lo3) | (values > hi3)).sum()),
    })

display(pd.DataFrame(diagnostic_rows))

,signal,min,max,IQR_1.5_flagged,IQR_3_flagged
0,X,-8.022103,1.360524,1,1
1,Y,-2.084078,1.552069,6683,1
2,Z,-1.042407,8.018888,15447,3
3,enmo,0.000000,7.033528,72528,33836


## 7. Final isolated-extreme correction

In [9]:
cols = ["X", "Y", "Z"]
df_iqr = [ts.copy() for ts in df]
outliers = []

for col in cols:
    values = np.concatenate([ts[col].dropna().values for ts in df])
    q1, q3 = np.percentile(values, [25, 75])
    iqr = q3 - q1
    lower, upper = q1 - 3 * iqr, q3 + 3 * iqr

    for ts_index, ts in enumerate(df):
        mask = ((ts[col] < lower) | (ts[col] > upper)) & ts[col].notna()
        for row_index in ts.index[mask]:
            outliers.append({
                "ts_index": ts_index,
                "user_id": ts["id"].iloc[0],
                "row_index": row_index,
                "column": col,
                "value": ts.loc[row_index, col],
                "lower_bound": lower,
                "upper_bound": upper,
            })
        df_iqr[ts_index].loc[mask, col] = np.nan

outliers_df = pd.DataFrame(outliers)

removed_enmo = []
for ts_index, ts in enumerate(df_iqr):
    mask = ts["enmo"] > 3
    for pos in np.where(mask.to_numpy())[0]:
        removed_enmo.append({
            "ts_index": ts_index,
            "user_id": ts["id"].iloc[0],
            "row_index": ts.index[pos],
            "original_value": ts["enmo"].iloc[pos],
        })
    ts.loc[mask, "enmo"] = np.nan

removed_enmo_df = pd.DataFrame(removed_enmo)

print("3×IQR X/Y/Z cells changed:", len(outliers_df))
display(outliers_df)
print("ENMO > 3 cells changed:", len(removed_enmo_df))
display(removed_enmo_df)

3×IQR X/Y/Z cells changed: 5


,ts_index,user_id,row_index,column,value,lower_bound,upper_bound
0,3012,1044,150,X,-8.022103,-3.592831,3.457558
1,119,3556,91,Y,-2.084078,-1.631648,1.637386
2,561,2064,59,Z,1.843629,-1.916247,1.703502
3,561,2064,60,Z,8.018888,-1.916247,1.703502
4,1862,858,155,Z,2.906080,-1.916247,1.703502


ENMO > 3 cells changed: 2


,ts_index,user_id,row_index,original_value
0,561,2064,60,7.027744
1,3012,1044,150,7.033528


## 8. Interpolate the seven missing cells created above

In [10]:
cols_to_fill = ["X", "Y", "Z", "enmo"]
n_created_missing = sum(ts[cols_to_fill].isna().sum().sum() for ts in df_iqr)

for ts in df_iqr:
    for col in cols_to_fill:
        ts[col] = ts[col].interpolate(method="linear", limit_direction="both")

remaining_missing = sum(ts[cols_to_fill].isna().sum().sum() for ts in df_iqr)

print("Missing cells to interpolate:", int(n_created_missing))
print("Remaining missing cells:", int(remaining_missing))

df = df_iqr

Missing cells to interpolate: 7
Remaining missing cells: 0


## 9. Non-wear impact and final conservative removal

Removing individual rows would destroy the common fixed length of 200 time steps.  
The final choice is therefore to remove every time series containing **at least one non-wear observation**.

In [11]:
nonwear_ratio = np.array([
    (ts["non-wear_flag"] == 1).mean()
    for ts in df
])

has_nonwear = nonwear_ratio > 0
affected_indices = np.flatnonzero(has_nonwear)

affected_ids = {
    df[i]["id"].iloc[0]
    for i in affected_indices
}

print("Time series with any non-wear:", int(has_nonwear.sum()))
print("Affected participants:", len(affected_ids))
print("Share of affected series:", 100 * has_nonwear.mean())
print(
    "Affected series with >50% non-wear:",
    int((nonwear_ratio[has_nonwear] > 0.50).sum()),
    "/",
    int(has_nonwear.sum()),
)

before_target = pd.Series(
    [int(ts["sii_binary"].iloc[0]) for ts in df],
    name="before"
).value_counts(normalize=True).sort_index()

# FINAL TRANSFORMATION
ts_selected = [
    ts.copy()
    for ts in df
    if not (ts["non-wear_flag"] == 1).any()
]

after_target = pd.Series(
    [int(ts["sii_binary"].iloc[0]) for ts in ts_selected],
    name="after"
).value_counts(normalize=True).sort_index()

print("\nRemoved:", len(df) - len(ts_selected))
print("Remaining:", len(ts_selected))
print("Participants remaining:", len({ts["id"].iloc[0] for ts in ts_selected}))

display(pd.concat([before_target, after_target], axis=1))

Time series with any non-wear: 179
Affected participants: 172
Share of affected series: 4.034257381113364
Affected series with >50% non-wear: 133 / 179

Removed: 179
Remaining: 4258
Participants remaining: 326


,proportion,proportion
0,0.666441,0.670503
1,0.333559,0.329497


## 10. Final dynamic channels

The retained dynamic signals are:

`X`, `Y`, `Z`, `enmo`, `light`

Excluded from modelling signals:
- `anglez`: strongly redundant with `Z`;
- `battery_voltage`: technical/device variable;
- `non-wear_flag`: constant after cleaning;
- `weekday`, `quarter`, `relative_date_PCIAT`: contextual and constant within each series;
- `id`: grouping variable;
- `sii_binary`: target only.

In [12]:
SIGNAL_COLS = ["X", "Y", "Z", "enmo", "light"]

final_all = pd.concat(ts_selected, ignore_index=True)
display(final_all[SIGNAL_COLS].describe().T)

,count,mean,std,min,25%,50%,75%,max
X,851600.0,-0.096891,0.545422,-1.465616,-0.578160,-0.242328,0.446603,1.360524
Y,851600.0,0.002391,0.354035,-1.484892,-0.232960,0.002007,0.238535,1.552069
Z,851600.0,-0.089596,0.400237,-1.042407,-0.353319,-0.113242,0.145677,1.699083
enmo,851600.0,0.058667,0.084546,0.000000,0.015257,0.032214,0.068701,2.520428
light,851600.0,54.359341,211.480804,0.000000,3.252416,8.409254,21.253337,2594.622070


## 11. Global z-score copy for clustering only

This scaling is fitted on the complete cleaned dataset because clustering is unsupervised.  
**Do not use this copy for supervised classification.**

In [13]:
from sklearn.preprocessing import StandardScaler

all_clean = pd.concat(ts_selected, ignore_index=True)

scaler_global = StandardScaler()
scaler_global.fit(all_clean[SIGNAL_COLS])

ts_selected_scaled_global = []

for ts in ts_selected:
    scaled = ts.copy()
    scaled[SIGNAL_COLS] = scaler_global.transform(ts[SIGNAL_COLS])
    ts_selected_scaled_global.append(scaled)

scaled_all = pd.concat(ts_selected_scaled_global, ignore_index=True)

display(
    scaled_all[SIGNAL_COLS]
    .agg(["mean","std","min","max"])
    .T
    .round(4)
)

,mean,std,min,max
X,-0.0,1.0,-2.5095,2.6721
Y,-0.0,1.0,-4.2009,4.3772
Z,0.0,1.0,-2.3806,4.4691
enmo,0.0,1.0,-0.6939,29.1175
light,0.0,1.0,-0.2570,12.0118


## 12. No general detrending or smoothing

The final pipeline does not apply general detrending or noise smoothing.  
Trend and short local peaks may carry behavioural information and are relevant to motif/discord analysis.

## 13. Save Module 3 preprocessing outputs

In [15]:
UNSCALED_PATH = TS_PROCESSED_DIR / "CMI_timeseries_dataset_preproc.pkl"
SCALED_PATH = TS_PROCESSED_DIR / "CMI_timeseries_dataset_preproc_scal.pkl"
SCALER_PATH = TS_PROCESSED_DIR / "global_clustering_scaler.pkl"

pd.to_pickle(ts_selected, UNSCALED_PATH)
pd.to_pickle(ts_selected_scaled_global, SCALED_PATH)

import joblib
joblib.dump(scaler_global, SCALER_PATH)

# Re-open to verify the materialized checkpoints.
unscaled_check = pd.read_pickle(UNSCALED_PATH)
scaled_check = pd.read_pickle(SCALED_PATH)

print("Saved:", UNSCALED_PATH)
print("Saved:", SCALED_PATH)
print("Saved:", SCALER_PATH)
print("Verified shapes:", len(unscaled_check), unscaled_check[0].shape, "|", len(scaled_check), scaled_check[0].shape)

Saved: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2/data/processed/timeseries/CMI_timeseries_dataset_preproc.pkl
Saved: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2/data/processed/timeseries/CMI_timeseries_dataset_preproc_scal.pkl
Saved: /Users/sarabiiavasco/Codice/DATA MINING/PROGETTO DM2/data/processed/timeseries/global_clustering_scaler.pkl
Verified shapes: 4258 (200, 13) | 4258 (200, 13)
